In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('Data/TrainData_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

print('Resized train vol_shape:', x_train.shape[1:])
print('Resized train shape:', x_train.shape)

Resized train vol_shape: (128, 256, 256)
Resized train shape: (400, 128, 256, 256)


In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        # ファインチューニングでは同じ症例同士のペアを避ける
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        while np.any(idx2 == idx1):
            same_case = idx2 == idx1
            idx2[same_case] = np.random.randint(0, x_data.shape[0], size=same_case.sum())
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [9]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\__init__.py
C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\torch\networks.py
['VxmDense', 'VxmDense1', 'VxmDense2', 'VxmDense_128_256', 'VxmDense_128_256_256']


In [10]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

[64, 128, 128]


C:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [38]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

# 80k curriculum pretraining with stage checkpoints and accuracy monitoring

+This version keeps the original pretraining loss and curriculum schedule, but saves a checkpoint every 2,000 epochs. It evaluates the model at its current curriculum displacement after every stage and compares 0/10/20/30/40-pixel translations at stages 1/10/20/30/40. No fine-tuning is run in this cell.


In [ ]:
# Curriculum pretraining: save one checkpoint and one evaluation record for every 2,000-epoch stage.
# This keeps the ORIGINAL pretraining objective:
#   100 * MSE(Moving', Moved) + 0.01 * MSE(teacher_DVF, predicted_DVF)
import csv
import time
from pathlib import Path
from IPython.display import display

TOTAL_EPOCHS = 80000
STAGE_EPOCHS = 2000
STAGE_CHECKPOINT_DIR = Path('curriculum_80k_stage_checkpoints')
STAGE_IMAGE_DIR = Path('curriculum_80k_stage_evaluation_images')
STAGE_CHECKPOINT_DIR.mkdir(exist_ok=True)
STAGE_IMAGE_DIR.mkdir(exist_ok=True)

# Each 2,000-epoch stage is evaluated at its own maximum displacement (1, 2, ..., 40 px).
# At the five named milestones, all five common shifts are also evaluated for direct comparison.
COMMON_TEST_SHIFTS = (0, 10, 20, 30, 40)
COMPARISON_STAGES = {1, 10, 20, 30, 40}
SHOW_MILESTONE_IMAGES = True
SAVE_EVERY_STAGE_IMAGE = True
RANDOM_SEED = 20260729
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def gaussian_smooth_3d(tensor, kernel_size=5, sigma=1.0):
    """Same Gaussian DVF smoothing used by the original 80k pretraining cell."""
    from scipy.ndimage import gaussian_filter
    smoothed = gaussian_filter(tensor.detach().cpu().numpy(), sigma=[0, 0, sigma, sigma, sigma])
    return torch.tensor(smoothed, dtype=torch.float32, device=tensor.device)

def curriculum_ncc(reference, prediction, eps=1e-6):
    reference = reference - reference.mean()
    prediction = prediction - prediction.mean()
    return ((reference * prediction).sum() /
            torch.sqrt(reference.square().sum() * prediction.square().sum()).clamp_min(eps)).item()

def curriculum_reconstruct(moving, fixed):
    moving_analysis = analysis_filter_3d(moving, analysis)
    fixed_analysis = analysis_filter_3d(fixed, analysis)
    moving_bands = down_sampling_3d(moving_analysis)
    fixed_bands = down_sampling_3d(fixed_analysis)
    predicted_flow = model3D(moving_bands, fixed_bands)
    moved_bands = torch.cat(
        [transformer(moving_bands[:, channel:channel + 1], predicted_flow)
         for channel in range(moving_bands.shape[1])], dim=1
    )
    moved, _ = synthesis_filter_3d(up_sampling_3d(moved_bands), synthesis_filters)
    return moved, predicted_flow

evaluation_moving = torch.from_numpy(x_train[0:1]).unsqueeze(1).to(device, dtype=torch.float32)
evaluation_slice = evaluation_moving.shape[2] // 2
evaluation_vmin, evaluation_vmax = np.percentile(evaluation_moving[0, 0].detach().cpu().numpy(), [1, 99])

def evaluate_known_x_shift(stage, test_shift, save_image=False, show_image=False):
    # x direction only: z/y displacements are zero. The expected input displacement is known exactly.
    known_full_flow = torch.zeros((1, 3, *evaluation_moving.shape[2:]), device=device)
    known_full_flow[:, 2] = float(test_shift)
    moving_prime = transformer256(evaluation_moving, known_full_flow)
    model3D.eval()
    with torch.no_grad():
        moved, predicted_flow = curriculum_reconstruct(evaluation_moving, moving_prime)

    mse_before = MSE_Loss(moving_prime, evaluation_moving).item()
    mse_after = MSE_Loss(moving_prime, moved).item()
    mae_before = (moving_prime - evaluation_moving).abs().mean().item()
    mae_after = (moving_prime - moved).abs().mean().item()
    rmse_before = float(np.sqrt(mse_before))
    rmse_after = float(np.sqrt(mse_after))
    ncc_before = curriculum_ncc(moving_prime, evaluation_moving)
    ncc_after = curriculum_ncc(moving_prime, moved)
    improvement = 100.0 * (mse_before - mse_after) / max(mse_before, 1e-12)
    row = {
        'stage': stage, 'epoch': stage * STAGE_EPOCHS, 'test_shift_x_px': test_shift,
        'mse_before': mse_before, 'mse_after': mse_after,
        'mae_before': mae_before, 'mae_after': mae_after,
        'rmse_before': rmse_before, 'rmse_after': rmse_after,
        'ncc_before': ncc_before, 'ncc_after': ncc_after,
        'mse_reduction_percent': improvement,
        'predicted_flow_abs_mean_lowres': predicted_flow.abs().mean().item(),
    }
    if save_image:
        images = [evaluation_moving[0, 0, evaluation_slice].detach().cpu().numpy(),
                  moving_prime[0, 0, evaluation_slice].detach().cpu().numpy(),
                  moved[0, 0, evaluation_slice].detach().cpu().numpy()]
        fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
        for axis, image, label_name in zip(axes, images, ['Moving', "Moving′", 'Moved']):
            axis.imshow(image, cmap='gray', vmin=evaluation_vmin, vmax=evaluation_vmax)
            axis.set_title(label_name)
            axis.axis('off')
        fig.suptitle(
            f'Curriculum stage {stage} ({stage * STAGE_EPOCHS:,} epochs): x shift {test_shift} px\n'
            f'MSE {mse_before:.6f} → {mse_after:.6f} ({improvement:+.1f}%), '
            f'RMSE {rmse_before:.6f} → {rmse_after:.6f}, NCC {ncc_before:.4f} → {ncc_after:.4f}'
        )
        image_path = STAGE_IMAGE_DIR / f'stage_{stage:02d}_epoch_{stage * STAGE_EPOCHS:05d}_xshift_{test_shift:02d}.png'
        fig.savefig(image_path, dpi=200, bbox_inches='tight')
        if show_image:
            plt.show()
        plt.close(fig)
    return row

def save_history(rows, filename):
    if rows:
        with open(filename, 'w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)

def save_progress_figure(stage_rows, comparison_rows):
    if not stage_rows:
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    stage_numbers = [row['stage'] for row in stage_rows]
    axes[0].plot(stage_numbers, [row['mse_after'] for row in stage_rows], marker='o', label='after')
    axes[0].plot(stage_numbers, [row['mse_before'] for row in stage_rows], linestyle='--', label='before')
    axes[0].set(title='Own-stage MSE', xlabel='Curriculum stage / maximum px'); axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].plot(stage_numbers, [row['ncc_after'] for row in stage_rows], marker='o', label='after')
    axes[1].plot(stage_numbers, [row['ncc_before'] for row in stage_rows], linestyle='--', label='before')
    axes[1].set(title='Own-stage NCC', xlabel='Curriculum stage / maximum px'); axes[1].legend(); axes[1].grid(alpha=.25)
    for shift in COMMON_TEST_SHIFTS:
        selected = [row for row in comparison_rows if row['test_shift_x_px'] == shift]
        if selected:
            axes[2].plot([row['stage'] for row in selected], [row['mse_after'] for row in selected], marker='o', label=f'{shift} px')
    axes[2].set(title='Milestone MSE after registration', xlabel='Curriculum stage'); axes[2].legend(); axes[2].grid(alpha=.25)
    fig.savefig(STAGE_IMAGE_DIR / 'curriculum_stage_accuracy_progress.png', dpi=200, bbox_inches='tight')
    plt.show(); plt.close(fig)

losses, loss_vecs, loss_images = [], [], []
stage_rows, comparison_rows = [], []
started_at = time.time()

for epoch in range(TOTAL_EPOCHS):
    stage = epoch // STAGE_EPOCHS + 1
    shift_range = stage
    train_batch, _ = next(train_generator)
    moving_images = torch.tensor(train_batch[0], dtype=torch.float32).to(device)

    # Same smooth random-DVF curriculum as the original notebook.
    B, D, H, W = 2, 8, 16, 16
    displacement_field = (torch.rand((B, 3, D, H, W), dtype=torch.float32) * 2 - 1).to(device) * shift_range
    displacement_field = gaussian_smooth_3d(displacement_field, sigma=2.0)
    displacement_field = torch.nn.functional.interpolate(displacement_field, size=(128, 256, 256), mode='trilinear', align_corners=False)
    displacement_field128 = torch.nn.functional.interpolate(displacement_field, size=(64, 128, 128), mode='trilinear', align_corners=False)
    moving_images2 = transformer256(moving_images, displacement_field)

    moving_w = down_sampling_3d(analysis_filter_3d(moving_images, analysis)).to(device)
    moving_images2_w = down_sampling_3d(analysis_filter_3d(moving_images2, analysis)).to(device)
    optimizer.zero_grad()
    Vec = model3D(moving_w, moving_images2_w)
    moving_warped = torch.cat([transformer(moving_w[:, i:i + 1], Vec) for i in range(moving_w.shape[1])], dim=1)
    transformed_image, _ = synthesis_filter_3d(up_sampling_3d(moving_warped), synthesis_filters)
    loss_vec = MSE_Loss(displacement_field128, Vec) * 0.01
    loss_image = MSE_Loss(moving_images2, transformed_image) * 100
    loss = loss_vec + loss_image
    loss.backward()
    optimizer.step()
    losses.append(loss.item()); loss_vecs.append(loss_vec.item()); loss_images.append(loss_image.item())

    if (epoch + 1) % 100 == 0:
        elapsed = time.time() - started_at
        eta_seconds = elapsed / (epoch + 1) * (TOTAL_EPOCHS - epoch - 1)
        print(f'Epoch {epoch + 1:05d}/{TOTAL_EPOCHS}: stage={stage:02d}, max=±{shift_range}px, '
              f'loss={loss.item():.6f}, image={loss_image.item():.6f}, dvf={loss_vec.item():.6f}, '
              f'ETA={eta_seconds / 3600:.1f} h')

    if (epoch + 1) % STAGE_EPOCHS == 0:
        model3D.eval()
        own_stage = evaluate_known_x_shift(stage, shift_range, save_image=SAVE_EVERY_STAGE_IMAGE, show_image=False)
        stage_rows.append(own_stage)
        if stage in COMPARISON_STAGES:
            for test_shift in COMMON_TEST_SHIFTS:
                comparison_rows.append(evaluate_known_x_shift(
                    stage, test_shift, save_image=True, show_image=SHOW_MILESTONE_IMAGES
                ))
        checkpoint = {
            'epoch': epoch + 1, 'stage': stage, 'max_shift_pixels': shift_range,
            'model_state_dict': model3D.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
            'training_loss': loss.item(), 'image_loss': loss_image.item(), 'dvf_loss': loss_vec.item(),
            'own_stage_evaluation': own_stage,
        }
        checkpoint_path = STAGE_CHECKPOINT_DIR / f'pretrain_curriculum_stage_{stage:02d}_epoch_{epoch + 1:05d}.pth'
        torch.save(checkpoint, checkpoint_path)
        save_history(stage_rows, STAGE_CHECKPOINT_DIR / 'curriculum_own_stage_metrics.csv')
        save_history(comparison_rows, STAGE_CHECKPOINT_DIR / 'curriculum_milestone_grid_metrics.csv')
        save_progress_figure(stage_rows, comparison_rows)
        print(f'Saved stage {stage:02d}: {checkpoint_path} | own-stage MSE={own_stage["mse_after"]:.6f}, NCC={own_stage["ncc_after"]:.4f}')

final_model_path = STAGE_CHECKPOINT_DIR / 'model_analysis_pipeline_pretrain_curriculum_final.pth'
torch.save(model3D.state_dict(), final_model_path)
print(f'Completed {TOTAL_EPOCHS:,} epochs. Final pretraining model: {final_model_path.resolve()}')
